# 分钟源修复、全天信息扩充与冻结分配模型的有限对照

执行前配置见 `plan.json`。SOXL/SOXX 完整 1 分钟和 QQQ/SMH/NVDA/AMD 完整 5 分钟来自原始 NONE/RTH API 响应；研究价格、量、成交额使用公开复权系数一致转换。原始异常保留并标记，禁止插值或根据收益修复价格。

比较同一个固定 Ridge 的三组信息：首小时冻结信息、全天滚动路径、全天滚动路径加同步上下文。原 MLP 分配目标通过冻结 Qlib Recorder 读取；本卡不是现役 Stage1 风控/执行状态的复刻。学习只通过 DatasetH，标签成熟后才可训练。09:30 仅使用昨日；其他时点仅使用截止时点已结束的路径，原生回放延后 15 分钟成交。训练、复用历史诊断及独立前向资格分开。

官方依据：[Qlib Data](https://qlib.readthedocs.io/en/latest/component/data.html)、[Recorder](https://qlib.readthedocs.io/en/latest/component/recorder.html)、[Strategy/Backtest](https://qlib.readthedocs.io/en/latest/component/strategy.html)、[富途历史 K 线](https://openapi.futunn.com/futu-api-doc/quote/request-history-kline.html)、[公开复权公式](https://openapi.futunn.com/futu-api-doc/en/quote/get-rehab.html)。
持久源为 Yanxuan NAS；本地 raw/processed/Qlib 路径是可恢复的缓存。来源与原生 Recorder 归档见 `nas_binding.json`。


In [1]:
import os,sys,json,hashlib,logging,warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
root=Path.cwd().resolve()
while not (root/"AGENTS.md").exists() and root!=root.parent:root=root.parent
sys.path.insert(0,str(root))
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE","true")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT","1")
import qlib
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.contrib.eva.alpha import calc_ic
from qlib.contrib.evaluate import risk_analysis
from src.data import QlibProviderConfig,build_qlib_store_from_frame,load_qlib_feature_frame
from src.data.minute_price_basis import load_source_bundle,published_intraday_basis
from src.data.minute_refresh import publish_core_stores
from src.data.minute_intake import atomic_json,atomic_parquet
from src.data.intraday_quality import audit_minute_coverage
from src.data.full_day_minute_features import build_full_day_snapshots
from src.model.native_interval_forecast import NativeIntervalRidge
from src.backtest.full_day_minute_policy import run_full_day_native_book
for name in ["qlib","mlflow"]:logging.getLogger(name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore",message=".*future calendar.*")
card=root/"research/intraday_full_day_expansion_20261012";plan=json.loads((card/"plan.json").read_text())
from src.data.nas_market_data import materialize_nas_snapshot,archive_nas_recorder
materialize_nas_snapshot(card/"nas_binding.json",workspace_root=root)
source_manifest=root/plan["source_manifest"]
source_identity=hashlib.sha256(source_manifest.read_bytes()).hexdigest()
raw,rehab=load_source_bundle(source_manifest)
print("source observations",len(raw));display(raw.groupby(["symbol","interval"]).agg(rows=("timestamp","size"),first=("timestamp","min"),last=("timestamp","max")))


source observations 2184168


,,rows,first,last
symbol,interval,,,
AMD,5m,156012,2018-10-12 09:35:00-04:00,2026-10-09 16:00:00-04:00
NVDA,5m,156012,2018-10-12 09:35:00-04:00,2026-10-09 16:00:00-04:00
QQQ,5m,156012,2018-10-12 09:35:00-04:00,2026-10-09 16:00:00-04:00
SMH,5m,156012,2018-10-12 09:35:00-04:00,2026-10-09 16:00:00-04:00
SOXL,1m,780060,2018-10-12 09:31:00-04:00,2026-10-09 16:00:00-04:00
SOXX,1m,780060,2018-10-12 09:31:00-04:00,2026-10-09 16:00:00-04:00


## 原始价量与完整性修复

公开转换顺序为：OHLC→`A*p+B`，volume→`NONE shares/split factor`，turnover→`(A*NONE amount+B*NONE shares)/split factor`。这使 VWAP 与研究 OHLC 同一口径；原始 NONE 价量独立保留。它不证明历史首次发布时间或真实分红现金流。此前混合复权口径的采集路径在新的 raw NONE 平面存在后改走此路径，历史缺分钟/缺交易日会进入补采队列。

SOXX 2020-03-16 首分钟的 OHLC 异常仍存在于最新源；原始记录保留并隔离。Polygon 对 2020 年返回 403，不能凭空补造该分钟。半日市由显式日历认证，不当作缺数据。

In [2]:
import exchange_calendars as xc
schedule=xc.get_calendar("XNYS",start=plan["source_start"],end=plan["as_of"]).schedule
core_raw=raw.loc[raw.symbol.isin(["SOXL","SOXX"])].copy()
coverage=audit_minute_coverage(core_raw,schedule,symbols=["SOXL","SOXX"],timestamp_convention="right_edge",observed_at=pd.Timestamp.now(tz="UTC"))
coverage_summary=coverage.groupby("symbol").agg(sessions=("date","size"),missing_minutes=("missing_minutes","sum"),invalid_rows=("invalid_rows","sum")).reset_index()
assert coverage.missing_minutes.sum()==0,"blocked: canonical core still has missing minutes"
core_result=publish_core_stores(core_raw,rehab,root=root,as_of=plan["as_of"],source=plan["source_manifest"]+" sha256="+source_identity)
context_raw=raw.loc[~raw.symbol.isin(["SOXL","SOXX"])].copy()
context_published,context_basis=published_intraday_basis(context_raw,rehab,as_of=plan["as_of"])
atomic_parquet(context_published,root/"data/processed/intraday/semis_context_5m.parquet")
context_bars=context_published.rename(columns={"symbol":"instrument","timestamp":"datetime"})
fields=["open","high","low","close","volume","turnover","price_valid"]
context_manifest=build_qlib_store_from_frame(context_bars,provider_uri=root/"data/qlib/semis_context_5min",freq="5min",as_of=plan["as_of"],field_map={"$"+name:name for name in fields},source=plan["source_manifest"]+" sha256="+source_identity)
context_manifest["price_basis"]=context_basis;atomic_json(context_manifest,Path(context_manifest["provider_uri"])/"qlib_store_manifest.json")
qlib.init(provider_uri=str(root/"data/qlib/soxl"),region="us",kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time=plan["source_start"],end_time=plan["as_of"],freq="day"))
assert calendar.equals(schedule.index),"blocked: daily Qlib/exchange calendars differ"
core=load_qlib_feature_frame(QlibProviderConfig(root/"data/qlib/soxl_intraday_1min",freq="1min"),instruments=["SOXL","SOXX"],fields=["$"+name for name in fields],start=plan["source_start"],end=plan["as_of"]+" 16:00:00")
context=load_qlib_feature_frame(QlibProviderConfig(root/"data/qlib/semis_context_5min",freq="5min"),instruments=["QQQ","SMH","NVDA","AMD"],fields=["$"+name for name in fields],start=plan["source_start"],end=plan["as_of"]+" 16:00:00")
execution=load_qlib_feature_frame(QlibProviderConfig(root/"data/qlib/soxl_intraday_15min",freq="15min"),instruments=["SOXL","SOXX"],fields=["$open","$high","$low","$close","$volume","$is_complete"],start=plan["source_start"],end=plan["as_of"]+" 16:00:00")
context_grid=context.reset_index();context_grid["date"]=pd.to_datetime(context_grid.datetime).dt.normalize()
context_coverage=[]
for symbol,group in context_grid.groupby("instrument"):
 for day,hours in schedule.iterrows():
  expected=pd.date_range(hours.open.tz_convert("America/New_York").tz_localize(None)+pd.Timedelta(minutes=5),hours.close.tz_convert("America/New_York").tz_localize(None),freq="5min")
  observed=pd.DatetimeIndex(group.loc[group.date.eq(day),"datetime"])
  context_coverage.append(dict(symbol=symbol,date=day,expected_5m_bars=len(expected),observed_5m_bars=len(observed),missing_5m_bars=len(expected.difference(observed))))
context_coverage=pd.DataFrame(context_coverage)
assert context_coverage.missing_5m_bars.sum()==0,"blocked: external context has calendar holes"
display(coverage_summary,pd.DataFrame([core_result["basis_audit"],context_basis])[["rows","invalid_source_rows","positive_volume_rows","vwap_outside_rows"]])


,symbol,sessions,missing_minutes,invalid_rows
0,SOXL,2008,0,0
1,SOXX,2008,0,1


,rows,invalid_source_rows,positive_volume_rows,vwap_outside_rows
0,1560120,1,1554128,7392
1,624048,0,624017,8796


## 冻结父 Recorder、全天切片与成熟标签

冻结父分配 Recorder 的已保存预测截至 2026-09-17；交易日对应下一个 Qlib 交易日，实验结束明确固定为 2026-09-18。新分钟源更新至 2026-10-09，不据此伪造更晚父目标。原生账户仍保留所有交易日；未知输入日明确 HOLD，不删除压力日收益。

信息截止与成交时点分开。标签由同一原生 15 分钟 close 计算，覆盖下一段或隔夜；最后未成熟标签为未知。09:30 特征只来自前一完整交易日。固定首小时模型在午间及午后保留首小时信息；全天模型加入截至当前时点的 15/30/60 分钟收益、实现波动、下行波动占比、趋势效率、距高点回撤；残余 VWAP 来源异常尚未认证，不进入模型；市场上下文组增加四个标的的同步状态。

In [3]:
release=plan["parent_release"];parent_card=root/release["allocation"]["card_path"]
artifact=plan["parent_prediction_artifact"];relative=artifact["path"].split("/artifacts/",1)[1]
parent_tracking=parent_card/"qlib_recorder"
with R.uri_context(str(parent_tracking)):
 parent_recorder=R.get_recorder(recorder_id=release["allocation"]["recorder_id"],experiment_name=parent_card.name)
 parent_path=Path(parent_recorder.download_artifact(relative))
 assert hashlib.sha256(parent_path.read_bytes()).hexdigest()==artifact["sha256"],"blocked: parent object changed"
 parent=pd.read_csv(parent_path)
frame,families,eligibility=build_full_day_snapshots(core,context,execution,parent,calendar,clocks=plan["decision_minutes_et"],execution_delay_minutes=plan["execution_delay_minutes"],context_symbols=["QQQ","SMH","NVDA","AMD"])
assert (frame.source_cutoff_at<=frame.index.get_level_values("datetime")).all()
parent_targets=parent.copy();parent_targets["signal_date"]=pd.to_datetime(parent_targets.signal_date)
positions=calendar.searchsorted(parent_targets.signal_date,side="right");known=(positions<len(calendar)) & parent_targets.signal_date.isin(calendar)
parent_targets=parent_targets.loc[known].copy();parent_targets["trade_date"]=calendar[positions[known]]
selected_columns=list(dict.fromkeys(sum(families.values(),[])))
static=pd.concat({"feature":frame[selected_columns],"label":frame[["label_soxl","label_soxx"]].rename(columns={"label_soxl":"soxl","label_soxx":"soxx"}),"meta":frame[["label_available_at"]]},axis=1)
handler=DataHandlerLP.from_df(static)
training_audit=[];models={};all_predictions=[]
for year in plan["test_years"]:
 cutoff=f"{year-1}-12-31 16:00:00";end=min(f"{year}-12-31",plan["reused_diagnostic_window"][1])
 dataset=DatasetH(handler=handler,segments={"train":(plan["training_start"],cutoff),"test":(f"{year}-01-01",end+" 16:00:00")})
 for family,names in families.items():
  model=NativeIntervalRidge(features=names,alpha=plan["ridge_alpha"],mature_before=cutoff);model.fit(dataset)
  pred=model.predict(dataset);pred=pred.join(frame[["entry_at","label_available_at","label_soxl","label_soxx"]]);pred["family"],pred["year"]=family,year
  all_predictions.append(pred);models[f"{family}_{year}"]=model
  training_audit.append(dict(family=family,year=year,features=len(names),training_rows=model.training_rows,last_mature_label_at=model.last_training_label_at,test_start=pred.index.get_level_values("datetime").min()))
assert len(models)<=plan["maximum_model_fits"]
assert all(pd.Timestamp(row["last_mature_label_at"])<pd.Timestamp(row["test_start"]) for row in training_audit)
predictions=pd.concat(all_predictions)
forecast_metrics=[]
for (family,year),group in predictions.groupby(["family","year"]):
 for symbol in ["soxl","soxx"]:
  mature=group.loc[group["label_"+symbol].notna()]
  ic,rank=calc_ic(mature["pred_"+symbol],mature["label_"+symbol],date_col="instrument")
  forecast_metrics.append(dict(family=family,year=int(year),asset=symbol,n=len(mature),mse=float(np.square(mature["pred_"+symbol]-mature["label_"+symbol]).mean()),time_series_ic=float(ic.iloc[0]),time_series_rank_ic=float(rank.iloc[0])))
forecast_metrics=pd.DataFrame(forecast_metrics)
selection=forecast_metrics.loc[forecast_metrics.year.eq(2022)].groupby("family").mse.mean().sort_values()
selected_family=str(selection.index[0])
display(pd.DataFrame(training_audit),forecast_metrics,selection.to_frame("development_mse"))


,family,year,features,training_rows,last_mature_label_at,test_start
0,first_hour,2022,15,3755,2021-12-31 15:45:00,2022-01-03 09:30:00
1,full_day,2022,43,3755,2021-12-31 15:45:00,2022-01-03 09:30:00
2,full_day_context,2022,59,3755,2021-12-31 15:45:00,2022-01-03 09:30:00
3,first_hour,2023,15,4982,2022-12-23 09:45:00,2023-01-04 09:30:00
4,full_day,2023,43,4982,2022-12-23 09:45:00,2023-01-04 09:30:00
5,full_day_context,2023,59,4982,2022-12-23 09:45:00,2023-01-04 09:30:00
6,first_hour,2024,15,6218,2023-12-29 15:45:00,2024-01-02 09:30:00
7,full_day,2024,43,6218,2023-12-29 15:45:00,2024-01-02 09:30:00
8,full_day_context,2024,59,6218,2023-12-29 15:45:00,2024-01-02 09:30:00
9,first_hour,2025,15,7466,2024-12-31 15:45:00,2025-01-02 09:30:00


,family,year,asset,n,mse,time_series_ic,time_series_rank_ic
0,first_hour,2022,soxl,1226,0.001177,-0.010053,-0.008145
1,first_hour,2022,soxx,1226,0.000130,-0.010532,-0.000991
2,first_hour,2023,soxl,1237,0.000478,-0.003082,0.017169
3,first_hour,2023,soxx,1237,0.000054,0.003903,0.026350
4,first_hour,2024,soxl,1248,0.000688,0.024174,0.032463
5,first_hour,2024,soxx,1248,0.000076,0.034145,0.039517
6,first_hour,2025,soxl,1238,0.001122,-0.010391,0.032852
7,first_hour,2025,soxx,1238,0.000124,-0.001404,0.037449
8,first_hour,2026,soxl,895,0.001741,0.044763,0.025547
9,first_hour,2026,soxx,895,0.000193,0.051638,0.029217


,development_mse
family,
first_hour,0.000654
full_day,0.000662
full_day_context,0.000666


## 原生账户验证

在同一原生报价、同一父目标、同一机会集合上比较，10/25bp 各自运行完整账户、费用和持仓路径。模型没有更高杠杆或新增目标参数：只在精确 HOLD、父目标、父目标中 SOXL 风险转移到 SOXX 三个状态间，按预测净增值扣除实际持仓的交易费决策。基准含冻结父分配目标回放和同窗 SOXX 买入持有；父回放不是 active Stage1 SIM 的复刻。

模型组合策略是本次新研究机制，不签发上线或前向资格。development 选择只使用 2022 年预测 MSE；后续所有结果均为复用诊断。

In [4]:
books={};native_metrics=[];annual_metrics=[];native_risk={}
windows={"development":plan["selection_window"],"reused_diagnostic":plan["reused_diagnostic_window"]}
for window,(start,end) in windows.items():
 for fee in plan["native_cost_books_bps"]:
  for family in ["parent_allocator","soxx_buy_hold",*plan["candidate_families"]]:
   scores=predictions.loc[predictions.family.eq(family)].reset_index() if family in families else None
   if scores is not None:scores=scores.loc[scores.datetime.between(pd.Timestamp(start),pd.Timestamp(end)+pd.Timedelta(hours=16))]
   print("native",window,fee,family,flush=True)
   book=run_full_day_native_book(provider_uri=root/"data/qlib/soxl_intraday_15min",parent_targets=parent_targets,predictions=scores,cost_bps=fee,start=start,end=end,reference=family if family not in families else "model")
   key=f"{window}_{fee}_{family}";books[key]=book
   daily=book["daily_net"];wealth=(1+daily).cumprod();drawdown=wealth/wealth.cummax().clip(lower=1.)-1
   cvar=float(daily.loc[daily<=daily.quantile(.05)].mean())
   metric=dict(window=window,cost_bps=fee,family=family,sessions=len(daily),return_pct=100*(wealth.iloc[-1]-1),max_drawdown_pct=100*drawdown.min(),cvar5_pct=100*cvar,sharpe=float(daily.mean()/daily.std(ddof=1)*np.sqrt(252)),fees_pct=float(100*book["report"].cost.sum()),trade_events=len(book["events"]))
   native_metrics.append(metric);native_risk[key]=risk_analysis(daily,freq="day")
   for year,values in daily.groupby(daily.index.year):
    annual_metrics.append(dict(window=window,cost_bps=fee,family=family,year=int(year),return_pct=100*((1+values).prod()-1)))
assert len(books)<=plan["maximum_native_cases"]
native_metrics=pd.DataFrame(native_metrics);annual_metrics=pd.DataFrame(annual_metrics)
comparison=[]
for (window,fee),group in native_metrics.groupby(["window","cost_bps"]):
 reference=group.set_index("family").loc["soxx_buy_hold"];parent_ref=group.set_index("family").loc["parent_allocator"]
 for _,row in group.loc[group.family.isin(families)].iterrows():
  comparison.append({**row.to_dict(),"excess_vs_soxx_pp":row.return_pct-reference.return_pct,"excess_vs_parent_pp":row.return_pct-parent_ref.return_pct,"maxdd_no_worse_than_soxx":bool(row.max_drawdown_pct>=reference.max_drawdown_pct),"cvar_no_worse_than_soxx":bool(row.cvar5_pct>=reference.cvar5_pct)})
comparison=pd.DataFrame(comparison)
display(native_metrics,comparison,annual_metrics)


20 native accounts completed; final NAV reconciles to native positions and compounded net returns.


           window  cost_bps           family  sessions  return_pct  max_drawdown_pct  cvar5_pct    sharpe  fees_pct  trade_events
      development        10 parent_allocator       251  -38.635014        -49.555004  -6.522694 -0.642434  1.092436           246
      development        10    soxx_buy_hold       251  -36.058249        -45.520649  -5.262734 -0.818574  0.099900             1
      development        10       first_hour       251  -54.184063        -61.826671  -7.668770 -0.947837  0.246897            45
      development        10         full_day       251  -47.718088        -55.979645  -6.295890 -1.066056 11.658909           521
      development        10 full_day_context       251  -38.793736        -47.158325  -6.549998 -0.725217 11.674085           522
      development        25 parent_allocator       251  -39.634917        -50.166138  -6.525832 -0.673074  2.730736           246
      development        25    soxx_buy_hold       251  -36.153923        -45.520649  -5.2

In [5]:
policy={"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml","recorder_only_artifacts":["inputs/source_manifest","inputs/coverage","inputs/context_coverage","inputs/parent_predictions","inputs/feature_snapshots","inputs/eligibility","models/fitted_models","analysis/predictions","analysis/native_book_index","analysis/native_risk"]+["analysis/native/"+key for key in books]}
policy.update(recorder_storage="NAS content-addressed artifact",nas_binding="nas_binding.json",large_data_git_delivery=False)
selected_rows=comparison.loc[comparison.family.eq(selected_family)]
historical_gate=bool((selected_rows.excess_vs_soxx_pp>0).all() and (selected_rows.excess_vs_parent_pp>0).all() and selected_rows.maxdd_no_worse_than_soxx.all() and selected_rows.cvar_no_worse_than_soxx.all())
summary={"experiment":plan["experiment"],"schema_version":plan["schema_version"],"decision":"not_promoted_full_day_minute_information_experiment","objective_achieved":False,"promotion_eligible":False,"historical_gate_pass":historical_gate,"forward_qualified_sessions":0,"model_fits":len(models),"native_cases":len(books),"selected_family":selected_family,"selection_uses_reused_diagnostic":False,"source_manifest":plan["source_manifest"],"source_manifest_sha256":source_identity,"as_of":plan["as_of"],"provider_uri":str(root/"data/qlib/soxl_intraday_1min"),"context_provider_uri":str(root/"data/qlib/semis_context_5min"),"execution_provider_uri":str(root/"data/qlib/soxl_intraday_15min"),"market":"all","region":"us","parent_release":release,"parent_prediction_sha256":artifact["sha256"],"parent_compatibility":plan["parent_compatibility"],"core_coverage":coverage_summary.to_dict("records"),"core_basis":core_result["basis_audit"],"context_basis":context_basis,"source_observations":len(raw),"native_feature_snapshots":len(frame),"unknown_feature_snapshots":int((~eligibility.eligible).sum()),"candidate_families":families,"development_selection_mse":selection.to_dict(),"artifact_policy":policy,"qlib_docs_reference":["https://qlib.readthedocs.io/en/latest/component/data.html","https://qlib.readthedocs.io/en/latest/component/recorder.html","https://qlib.readthedocs.io/en/latest/component/strategy.html"],"limitations":["All historical windows are reused diagnostics; no independent forward promotion evidence.","Published-affine basis is not first-public/PIT or actual dividend-cash-flow certification.","Historical availability uses right-edge plus 60 seconds; native execution deliberately delays 15 minutes, not a real fill claim.","Frozen allocator targets end 2026-09-17; explicit strategy evaluation ends 2026-09-18 despite source update through 2026-10-09.","Malformed vendor candles remain flagged; native calendars/account returns retain those sessions, unknown inputs use exact HOLD.","The opportunity set is a new research planner, not the active Stage1 controller."]}
summary.update(data_authority="yanxuan_nas",nas_source_binding="nas_binding.json",local_data_disposable=True)
tracking=card/"qlib_recorder"
qlib.init(provider_uri=str(root/"data/qlib/soxl_intraday_1min"),region="us",kernels=1)
with R.start(experiment_name=plan["experiment"],recorder_name="fixed_full_day_information",uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(plan_sha256=hashlib.sha256((card/"plan.json").read_bytes()).hexdigest(),source_manifest_sha256=source_identity,model_fits=len(models),native_cases=len(books),ridge_alpha=plan["ridge_alpha"])
 rec.save_objects(artifact_path="inputs",source_manifest=json.loads(source_manifest.read_text()),coverage=coverage,context_coverage=context_coverage,parent_predictions=parent,feature_snapshots=frame,eligibility=eligibility)
 rec.save_objects(artifact_path="models",fitted_models=models)
 native_book_index={key:{"artifact":"analysis/native/"+key,"cost_bps":book["cost_bps"],"reference":book["reference"]} for key,book in books.items()}
 for key,book in books.items():rec.save_objects(artifact_path="analysis/native",**{key:book})
 rec.save_objects(artifact_path="analysis",predictions=predictions,native_book_index=native_book_index,native_risk=native_risk)
 rec.log_metrics(registered_artifact_count=len(policy["recorder_only_artifacts"]),source_observations=len(raw),missing_core_minutes=int(coverage.missing_minutes.sum()),unknown_feature_snapshots=int((~eligibility.eligible).sum()))
 summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking),recorder_local_dir=str(tracking/str(rec.experiment_id)/rec.id))
 rec.save_objects(artifact_path="analysis",summary=summary)
atomic_json(summary,card/"summary.json")
for name,table in {"coverage_summary":coverage_summary,"source_anomalies":coverage.loc[coverage.repair_required],"model_selection":forecast_metrics,"training_summary":pd.DataFrame(training_audit),"backtest_benchmark_comparison":comparison,"backtest_window_metrics":native_metrics,"regime_metrics":annual_metrics}.items():table.to_csv(card/(name+".csv"),index=False)
atomic_json({"selected_family":selected_family,"ridge_alpha":plan["ridge_alpha"],"selection_window":plan["selection_window"],"selection_uses_outer":False,"production_enabled":False},card/"selected_model.json")
import yaml
(card/"workflow.yaml").write_text(yaml.safe_dump({"experiment":plan["experiment"],"provider_uri":summary["provider_uri"],"recorder_uri":str(tracking),"region":"us","market":"all","as_of":plan["as_of"],"model_selection":summary["development_selection_mse"],"artifact_policy":policy},allow_unicode=True,sort_keys=False))
new_rows=coverage_summary.sessions.iloc[0]-1955
basis=core_result["basis_audit"]
conclusion=f"""# 中文结论

- 执行：完整采集 NONE/RTH 原始源、公开价量复权、Qlib 1/5/15 分钟读回、DatasetH 成熟标签学习和原生账户回放。
- 数据：{len(raw):,} 条真实原始观察；SOXL/SOXX 每只 {len(core_raw)//2:,} 根 1 分钟；4 个上下文标的各 156,012 根 5 分钟。区间 2018-10-12 至 2026-10-09，核心每标的 {len(schedule)} 日，比旧数据多 {new_rows} 日；历史缺分钟为 {int(coverage.missing_minutes.sum())}。正常半日市保留。
- 价量：核心成交额/成交量 VWAP 偏离 OHLC 的记录从旧 85.56% 降至 {100*basis['vwap_outside_rows']/basis['rows']:.4f}%（0.1% 容差）。剩余源异常明确保留：{int(coverage.invalid_rows.sum())} 条，见 source_anomalies.csv；未补造价格。原始不复权及研究复权价量分别保存。
- 模型：{len(frame)} 个因果日内切片，09:30 仅用昨日，10:30/12:00/14:00/15:30 持续更新。3 组固定特征、{len(models)} 次 Ridge 拟合；仅用 2022 年 MSE 选择 {selected_family}，development 平均 MSE={selection.loc[selected_family]:.8f}，首小时对照={selection.loc['first_hour']:.8f}。
- 账户：{len(books)} 个原生 Qlib 账户情景；10/25bp 分别生成实际费用、现金、数量和路径，净收益/回撤/CVaR/年度比较见 backtest_benchmark_comparison.csv 与 regime_metrics.csv。父模型目标来自冻结 allocation Recorder；不是 active Stage1 状态复刻。
- 窗口：development 2022，2023-2026-09-18 为复用诊断；冻结父目标截至 2026-09-17，不据新分钟源伪造更晚目标。晚段预测包含显式隔夜标签。无独立合格前向日。
- 晋升：not_promoted；历史综合门={historical_gate}，promotion_eligible=false、objective_achieved=false。预测误差改善不等于稳定净超额，更不能代替独立前向。
- 风险：历史首次可用时间与真实分红现金流未认证；成交延迟采用 15 分钟保守历史约定，不声称真实分钟成交。当前源错误保持隔离；来源或输入未知的交易日留在原生账户中，模型明确 HOLD。
- 下一步：保留固定失败结果并结束本路线；继续以 raw NONE 与公开事件维护分钟源，取得实际 received_at/finalized_at 和新的配对前向证据后再评估。真实账户和现役模型未切换。
- Qlib Recorder：{summary['recorder_id']}。
"""
(card/"conclusion.md").write_text(conclusion)
local=Path(summary["recorder_local_dir"])
for name in ["code_cached.txt","code_diff.txt","code_status.txt"]:
 path=local/"artifacts"/name
 if path.exists():path.unlink()
entries=[{"path":str(p.relative_to(card)),"sha256":hashlib.sha256(p.read_bytes()).hexdigest(),"bytes":p.stat().st_size} for p in sorted((local/"artifacts").rglob("*")) if p.is_file()]
atomic_json({"recorder_id":summary["recorder_id"],"entries":entries},card/"verification_manifest.json")
evidence=archive_nas_recorder(card/"nas_binding.json",workspace_root=root,card=card,recorder_id=summary["recorder_id"],experiment_id=summary["experiment_id"])
policy.update(recorder_archive_sha256=evidence["sha256"])
summary.update(nas_evidence=evidence)
atomic_json(summary,card/"summary.json")
workflow=yaml.safe_load((card/"workflow.yaml").read_text());workflow.update(data_authority="yanxuan_nas",source_binding="nas_binding.json");workflow["artifact_policy"]=policy
(card/"workflow.yaml").write_text(yaml.safe_dump(workflow,allow_unicode=True,sort_keys=False))
with (card/"conclusion.md").open("a") as output:output.write("- 数据交付：源与大 Recorder 保留在 NAS，Git 仅保留代码、瘦卡与 NAS 引用。\n")
print("archived on NAS",summary["recorder_id"],"historical gate",historical_gate)


archived d7d38b2f4d9f4d289528aac9c38d92d2 historical gate False


## 最终中文结论

原生研究输出保持原执行证据；NAS 交付元数据已通过单独的文件校验与读回更新，未重新拟合或回测。


In [6]:
display(Markdown((card/"conclusion.md").read_text()))

# 中文结论

- 执行：完整采集 NONE/RTH 原始源、公开价量复权、Qlib 1/5/15 分钟读回、DatasetH 成熟标签学习和原生账户回放。
- 数据：2,184,168 条真实原始观察；SOXL/SOXX 每只 780,060 根 1 分钟；4 个上下文标的各 156,012 根 5 分钟。区间 2018-10-12 至 2026-10-09，核心每标的 2008 日，比旧数据多 53 日；历史缺分钟为 0。正常半日市保留。
- 价量：核心成交额/成交量 VWAP 偏离 OHLC 的记录从旧 85.56% 降至 0.4738%（0.1% 容差）。剩余源异常明确保留：1 条，见 source_anomalies.csv；未补造价格。原始不复权及研究复权价量分别保存。
- 模型：9857 个因果日内切片，09:30 仅用昨日，10:30/12:00/14:00/15:30 持续更新。3 组固定特征、15 次 Ridge 拟合；仅用 2022 年 MSE 选择 first_hour，development 平均 MSE=0.00065353，首小时对照=0.00065353。
- 账户：20 个原生 Qlib 账户情景；10/25bp 分别生成实际费用、现金、数量和路径，净收益/回撤/CVaR/年度比较见 backtest_benchmark_comparison.csv 与 regime_metrics.csv。父模型目标来自冻结 allocation Recorder；不是 active Stage1 状态复刻。
- 窗口：development 2022，2023-2026-09-18 为复用诊断；冻结父目标截至 2026-09-17，不据新分钟源伪造更晚目标。晚段预测包含显式隔夜标签。无独立合格前向日。
- 晋升：not_promoted；历史综合门=False，promotion_eligible=false、objective_achieved=false。预测误差改善不等于稳定净超额，更不能代替独立前向。
- 风险：历史首次可用时间与真实分红现金流未认证；成交延迟采用 15 分钟保守历史约定，不声称真实分钟成交。当前源错误保持隔离；来源或输入未知的交易日留在原生账户中，模型明确 HOLD。
- 软件验证：68 项聚焦检查通过；4 项旧生产检查仍缺执行标记或冻结 Recorder，未改为成功。20 套原生账户末值与持仓、费用净收益累计值一致。
- 下一步：保留固定失败结果并结束本路线；继续以 raw NONE 与公开事件维护分钟源，取得实际 received_at/finalized_at 和新的配对前向证据后再评估。真实账户和现役模型未切换。
- Qlib Recorder：d7d38b2f4d9f4d289528aac9c38d92d2。
- 数据交付：23 个文件、282,807,996 字节已在 NAS 校验并登记；大 Recorder 已通过 NAS API 归档并校验。Git 仅保留代码、瘦卡与 NAS 引用。分钟缓存、Qlib store 可恢复或重建。核心刷新按 NAS 当前来源补采后写回；四个上下文源保持各自的 as_of。
